# 10 · Recuperación híbrida: TF-IDF + embeddings + RDF

**Etapa experimental y reproducible. No incluye un LLM ni genera afirmaciones históricas.**

**Cómo usar:** abre el notebook desde `notebooks/` y selecciona **Run → Run All Cells**. No requiere mover archivos ni ejecutar las etapas anteriores. Lee automáticamente `data/output/etapa7.zip`, `etapa8.zip` y, si existe, `etapa9.zip` (para los juicios humanos). Al terminar crea `data/output/etapa10.zip`.

Compara cuatro configuraciones de recuperación sobre **las mismas 10 fichas y 12 preguntas**:

- **A · TF-IDF:** baseline léxico de la etapa 9 (palabras 70 % + caracteres 30 %).
- **B · Embeddings:** búsqueda semántica con `paraphrase-multilingual-MiniLM-L12-v2`.
- **C · Híbrido:** fusión de rankings léxico y semántico mediante **Reciprocal Rank Fusion (RRF)**.
- **D · Híbrido + RDF:** el método C con una bonificación pequeña por co-membresía catalográfica **específica** (14 conexiones de etapa 8).

Los embeddings se calculan sobre **metadatos**, no sobre el texto íntegro de los PDF. La primera ejecución descargará el modelo desde Hugging Face y requiere Internet; posteriormente se usa su caché local. **No se requiere OCR, FAISS, Colab ni widgets.**

La co-membresía RDF **no es un hecho histórico**. Los resultados se presentan como candidatos documentales, no como respuestas sobre el contenido de los libros.

## 0. Dependencias

La celda siguiente instala solamente los paquetes que falten en el **entorno del kernel**. Si el modelo no puede descargarse, se detendrá con un error explícito; no sustituiremos embeddings reales por vectores ficticios.

In [1]:
import importlib.util, subprocess, sys
paquetes = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.3",
    "sentence_transformers": "sentence-transformers>=3.0,<7",
}
faltantes = [p for mod, p in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltantes:
    print("Instalando dependencias faltantes:", faltantes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()
print("Dependencias listas.")

Dependencias listas.


## 1. Leer resultados anteriores automáticamente

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import io, json, re, math, hashlib, zipfile, unicodedata, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

CWD=Path.cwd().resolve()
if (CWD/'data').is_dir() and (CWD/'notebooks').is_dir():
    ROOT=CWD
elif CWD.name.lower()=='notebooks' and (CWD.parent/'data').is_dir():
    ROOT=CWD.parent
else:
    raise FileNotFoundError("Abre este notebook desde la carpeta notebooks/ del proyecto memoria_chilena")

DIR_OUTPUT=ROOT/'data'/'output'
SALIDA=DIR_OUTPUT/'etapa10'; SALIDA.mkdir(parents=True, exist_ok=True)

def localizar_etapa(n):
    for p in [DIR_OUTPUT/f'etapa{n}.zip', ROOT/'data'/'input'/f'etapa{n}.zip']:
        if p.is_file(): return p
    return None

Z7=localizar_etapa(7); Z8=localizar_etapa(8); Z9=localizar_etapa(9)
if Z7 is None or Z8 is None:
    raise FileNotFoundError("Faltan etapa7.zip o etapa8.zip en data/output/. Conserva los resultados previos.")

def leer_csv_zip(ruta, nombre):
    with zipfile.ZipFile(ruta) as z:
        return pd.read_csv(io.BytesIO(z.read(nombre)), dtype=str, keep_default_na=False)

catalogo=leer_csv_zip(Z7,'catalogo_para_busqueda.csv')
relaciones=leer_csv_zip(Z8,'relaciones_filtradas.csv')
plantilla8=leer_csv_zip(Z8,'PLANTILLA_EVALUACION_MANUAL.csv')
ids=catalogo['mcid'].tolist()
consultas=list(dict.fromkeys(plantilla8['pregunta'].tolist()))
assert len(ids)==10 and len(set(ids))==10, "Se esperaban 10 documentos con MCID único"
assert len(consultas)==12, "Se esperaban 12 consultas de la etapa 8"
assert len(relaciones)==14, "Se esperaban 14 enlaces específicos de la etapa 8"
assert set(relaciones['mcid_origen']).union(relaciones['mcid_destino']).issubset(set(ids))

print('Raíz:',ROOT)
print(f'Documentos: {len(ids)} | Consultas: {len(consultas)} | Relaciones específicas: {len(relaciones)}')
print('Archivos:', Z7.name, Z8.name, (Z9.name if Z9 else 'etapa9 no disponible'))

Raíz: C:\Users\jvera\Documents\memoria_chilena
Documentos: 10 | Consultas: 12 | Relaciones específicas: 14
Archivos: etapa7.zip etapa8.zip etapa9.zip


## 2. Preparar textos catalográficos y reproducir el baseline léxico

El baseline TF-IDF replica la normalización, el corpus ponderado y las matrices de palabras/caracteres utilizados en 09. Para los **embeddings**, se crea una representación legible en la que se anteponen temas y títulos; se excluye la etiqueta universal `religiosidadandina` porque la comparten las diez fichas.

Las fechas se conservan como valores catalográficos sin deducir automáticamente años de publicación (por ejemplo, `start=900;end=900`).

In [3]:
ALIASES={'quichua':'quechua','piru':'peru','yncas':'incas'}

def normalizar(s):
    s=unicodedata.normalize('NFKD',str(s).lower())
    s=''.join(c for c in s if not unicodedata.combining(c))
    s=re.sub(r'[^a-z0-9\s]',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def expandir_alias(s):
    s=normalizar(s)
    for a,b in ALIASES.items():
        s=re.sub(r'\b'+a+r'\b',b,s)
    return s

def separar_unidades(s):
    return {u.strip() for u in str(s).split('|') if u.strip()}

# Excluir exclusivamente el identificador universal al calcular similitud/expansión.
UNIDAD_UNIVERSAL={'religiosidadandina'}
por_doc={r['mcid']: separar_unidades(r['unidades']) for _,r in catalogo.iterrows()}

def texto_lexico(r):
    partes=[]
    for col,n in [('titulo',4),('autor',2),('materias',1),('notas',2),('datos',1)]:
        partes.extend([expandir_alias(r[col])]*n)
    partes.extend([expandir_alias(' '.join(sorted(por_doc[r['mcid']]-UNIDAD_UNIVERSAL)))]*2)
    return ' '.join(partes)

corpus_lexico=[texto_lexico(r) for _,r in catalogo.iterrows()]
vec_p=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,token_pattern=r'(?u)\b\w\w+\b')
vec_c=TfidfVectorizer(analyzer='char',ngram_range=(3,5),sublinear_tf=True)
Xp=vec_p.fit_transform(corpus_lexico)
Xc=vec_c.fit_transform(corpus_lexico)

def puntuaciones_lexicas(pregunta):
    q=expandir_alias(pregunta)
    palabra=cosine_similarity(vec_p.transform([q]),Xp)[0]
    caracter=cosine_similarity(vec_c.transform([q]),Xc)[0]
    return .7*palabra+.3*caracter

ETIQUETAS_UNIDAD={
    'elpuebloaymara':'pueblo aymara',
    'cronistasperuanos':'cronistas peruanos',
    'geografiaculturaldechile':'geografía cultural de Chile',
    'bailesfolcloricoschilenos':'bailes folclóricos chilenos',
    'elsacramentodelapenintenciaenchilecolonial':'sacramento de la penitencia en Chile colonial',
    'iniciosydesarrollodelpensamientopedagogicoenchile':'pensamiento pedagógico en Chile',
    'lacompaniadejesusenchile':'Compañía de Jesús en Chile',
    'lasmisionesjesuitasdeparaguay,moxos,chiquitosymaynas':'misiones jesuitas de Paraguay Moxos Chiquitos y Maynas',
    'elserviciopostalenChilesiglosxvixxi':'servicio postal en Chile',
    'historiografiacolonialamericana':'historiografía colonial americana',
    'rutasdepoblamientoyprimerasadaptacioneshumanasenchile':'poblamiento y adaptaciones humanas en Chile',
}

def texto_embedding(r):
    temas=[ETIQUETAS_UNIDAD.get(u,u) for u in sorted(por_doc[r['mcid']]-UNIDAD_UNIVERSAL)]
    # El modelo MiniLM acepta textos cortos; anteponemos los campos más discriminantes.
    campos=[f'Temas: {"; ".join(temas)}' if temas else '',
            f'Título: {r["titulo"][:320]}',
            f'Autor: {r["autor"][:120]}' if r['autor'] else '',
            f'Materias: {r["materias"][:140]}',
            f'Notas: {r["notas"][:340]}' if r['notas'] else '',
            f'Datos bibliográficos: {r["datos"][:220]}' if r['datos'] else '']
    return ' '.join(x for x in campos if x)

corpus_semantico=[texto_embedding(r) for _,r in catalogo.iterrows()]
docs_idx=catalogo[['mcid','titulo','autor','fecha','materias','unidades','ficha']].copy()
docs_idx['texto_embedding']=corpus_semantico
docs_idx.to_csv(SALIDA/'documentos_indexados.csv',index=False,encoding='utf-8-sig')
print('Baseline léxico y 10 textos para embeddings preparados.')

Baseline léxico y 10 textos para embeddings preparados.


## 3. Generar embeddings reales

Modelo: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384 dimensiones). Calculamos similitud coseno con vectores normalizados; no necesitamos FAISS para diez documentos. Los embeddings de documentos se guardan con un hash para reutilizarlos en posteriores ejecuciones del mismo corpus.

In [4]:
from sentence_transformers import SentenceTransformer
MODELO_ID='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
print('Cargando modelo semántico:', MODELO_ID)
try:
    modelo=SentenceTransformer(MODELO_ID, device='cpu')
except Exception as e:
    raise RuntimeError('No se pudo cargar el modelo real. Verifica Internet en la primera ejecución y vuelve a ejecutar esta celda. No se generarán embeddings falsos.') from e

HASH_CORPUS=hashlib.sha256(json.dumps({'ids':ids,'textos':corpus_semantico,'modelo':MODELO_ID},ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
RUTA_EMB=SALIDA/'embeddings_documentos.npy'
RUTA_CACHE=SALIDA/'embeddings_cache.json'
cache_ok=False
if RUTA_EMB.exists() and RUTA_CACHE.exists():
    meta=json.loads(RUTA_CACHE.read_text(encoding='utf-8'))
    if meta.get('sha256_corpus')==HASH_CORPUS and meta.get('modelo')==MODELO_ID:
        emb_docs=np.load(RUTA_EMB,allow_pickle=False)
        cache_ok=emb_docs.shape==(len(ids),384) and np.isfinite(emb_docs).all()
if not cache_ok:
    emb_docs=np.asarray(modelo.encode(corpus_semantico,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False,batch_size=10),dtype=np.float32)
    assert emb_docs.shape==(len(ids),384), f'Dimensiones inesperadas: {emb_docs.shape}'
    np.save(RUTA_EMB,emb_docs,allow_pickle=False)
    RUTA_CACHE.write_text(json.dumps({'modelo':MODELO_ID,'sha256_corpus':HASH_CORPUS,'dimensiones':384,'documentos':len(ids)},indent=2),encoding='utf-8')

emb_consultas=np.asarray(modelo.encode(consultas,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False,batch_size=12),dtype=np.float32)
assert emb_consultas.shape==(len(consultas),384)
SCORES_SEMANTICOS=emb_consultas @ emb_docs.T
assert SCORES_SEMANTICOS.shape==(len(consultas),len(ids))
print('Embeddings reales listos. Caché reutilizada:',cache_ok)

Cargando modelo semántico: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embeddings reales listos. Caché reutilizada: False


## 4. Cuatro rankings (A/B/C/D)

**RRF** suma `1 / (60 + posición)` para los rankings léxico y semántico; evita mezclar directamente escalas de coseno y TF-IDF. Las coincidencias léxicas con puntaje cero no aportan voto léxico. Después, para el método D, se agrega una bonificación acotada **solo a los vecinos RDF específicos de los tres documentos mejor posicionados** en el método C. El conjunto de diez fichas candidatas no cambia.

Esta variante de RDF está pensada como **hipótesis a evaluar**, no como mejora garantizada ni como inferencia histórica.

In [5]:
posiciones={m:i for i,m in enumerate(ids)}
A=np.zeros((len(ids),len(ids)),dtype=float)
for r in relaciones.itertuples(index=False):
    i=posiciones[r.mcid_origen]; j=posiciones[r.mcid_destino]
    w=float(r.peso_idf)
    if i!=j and w>0:
        A[i,j]=A[j,i]=max(A[i,j],w)
assert sum(np.count_nonzero(A[i,j]) for i in range(len(ids)) for j in range(i+1,len(ids)))==14
# Cada fila indica cómo distribuir la evidencia del nodo entre vecinos específicos.
filas=A.sum(axis=1)
P=np.divide(A,filas[:,None],out=np.zeros_like(A),where=filas[:,None]>0)

RRF_K=60
BONUS_RDF=.10
TOP_SEMILLAS=3

def ordenar(scores):
    return sorted(range(len(ids)),key=lambda i:(-float(scores[i]),ids[i]))

def puntuaciones_rrf(lex,sem):
    resultado=np.zeros(len(ids),dtype=float)
    orden_lex=[i for i in ordenar(lex) if lex[i]>1e-12]
    orden_sem=ordenar(sem)
    for orden in (orden_lex,orden_sem):
        for puesto,i in enumerate(orden,1):
            resultado[i]+=1.0/(RRF_K+puesto)
    return resultado

def puntuaciones_rdf(fusion):
    norm=fusion/max(float(fusion.max()),1e-12)
    semilla=np.zeros(len(ids),dtype=float)
    for i in ordenar(fusion)[:TOP_SEMILLAS]:
        semilla[i]=norm[i]
    vecindad=P@semilla
    return norm+BONUS_RDF*vecindad,vecindad

rankings=[]
for qi,pregunta in enumerate(consultas):
    lex=puntuaciones_lexicas(pregunta)
    sem=SCORES_SEMANTICOS[qi].astype(float)
    fusion=puntuaciones_rrf(lex,sem)
    grafico,aporte=puntuaciones_rdf(fusion)
    for metodo,scores in [('A_tfidf',lex),('B_embeddings',sem),('C_hibrido_rrf',fusion),('D_hibrido_rdf',grafico)]:
        for puesto,i in enumerate(ordenar(scores),1):
            rankings.append({'pregunta':pregunta,'metodo':metodo,'posicion':puesto,
                 'mcid':ids[i],'titulo':catalogo.iloc[i]['titulo'],'ficha':catalogo.iloc[i]['ficha'],
                 'score_metodo':round(float(scores[i]),8),
                 'score_tfidf':round(float(lex[i]),8),'score_embedding_coseno':round(float(sem[i]),8),
                 'score_rrf':round(float(fusion[i]),8),
                 'aporte_vecindad_rdf':round(float(aporte[i]),8)})
RANK=pd.DataFrame(rankings)
assert len(RANK)==len(consultas)*4*len(ids)==480
assert RANK.groupby(['pregunta','metodo'])['mcid'].nunique().eq(len(ids)).all()
RANK.to_csv(SALIDA/'rankings_cuatro_metodos.csv',index=False,encoding='utf-8-sig')
print(f'{len(RANK)} filas de ranking: 12 preguntas × 4 métodos × 10 documentos.')

480 filas de ranking: 12 preguntas × 4 métodos × 10 documentos.


## 5. Diagnóstico de cambios de ranking (sin asumir relevancia)

In [6]:
def obtener_orden(q,m):
    return RANK.loc[(RANK.pregunta==q)&(RANK.metodo==m)].sort_values('posicion').mcid.tolist()

resumen=[]
for q in consultas:
    a=obtener_orden(q,'A_tfidf')
    for m in ['B_embeddings','C_hibrido_rrf','D_hibrido_rdf']:
        b=obtener_orden(q,m)
        resumen.append({'pregunta':q,'comparado_con':'A_tfidf','metodo':m,
            'top1_baseline':a[0],'top1_metodo':b[0],
            'top1_cambia':a[0]!=b[0],
            'orden_cambia':a!=b,
            'top5_cambia':set(a[:5])!=set(b[:5])})
MOV=pd.DataFrame(resumen)
MOV.to_csv(SALIDA/'diagnostico_cambios.csv',index=False,encoding='utf-8-sig')
print('Cambios de ranking respecto de TF-IDF (NO métricas de relevancia):')
display(MOV.groupby('metodo')[['top1_cambia','orden_cambia','top5_cambia']].sum())

Cambios de ranking respecto de TF-IDF (NO métricas de relevancia):


,top1_cambia,orden_cambia,top5_cambia
metodo,,,
B_embeddings,6,12,11
C_hibrido_rrf,5,12,8
D_hibrido_rdf,6,12,10


## 6. Evaluación humana opcional, sin inventar etiquetas

Se leen los juicios de la **etapa 9** (120 pares pregunta–documento) si están disponibles. No se modifican ni se rellenan automáticamente. **Solo si los 120 juicios tienen valores 0, 1 o 2** se calculan Precision@5, Recall@5, nDCG@5 y MRR@10 para los cuatro métodos.

Si faltan juicios, se guarda el estado *pendiente de evaluación*. Para calificarlos, utiliza el notebook **09 corregido sin widgets** (`evaluar_siguiente()`). Luego vuelve a ejecutar este notebook 10.

In [7]:
def leer_juicios():
    # Priorizar CSV de trabajo, que puede contener calificaciones posteriores a etapa9.zip.
    csv9=DIR_OUTPUT/'etapa9'/'juicios_relevancia.csv'
    if csv9.is_file():
        return pd.read_csv(csv9,dtype=str,keep_default_na=False),'CSV editable etapa9'
    if Z9 is not None:
        with zipfile.ZipFile(Z9) as z:
            if 'juicios_relevancia.csv' in z.namelist():
                return pd.read_csv(io.BytesIO(z.read('juicios_relevancia.csv')),dtype=str,keep_default_na=False),'ZIP etapa9'
    return None, 'no encontrados'

qrels,fuente_juicios=leer_juicios()
validos=0;pendientes=120;metricas_creadas=False
if qrels is not None:
    claves=list(zip(qrels['pregunta'],qrels['mcid']))
    esperadas={(q,i) for q in consultas for i in ids}
    assert len(claves)==120 and set(claves)==esperadas and len(set(claves))==120, 'Juicios 09 no coinciden con las 120 parejas'
    qrels.to_csv(SALIDA/'juicios_relevancia_lectura.csv',index=False,encoding='utf-8-sig')
    validos=int(qrels['relevancia_0_1_2'].isin(['0','1','2']).sum())
    pendientes=120-validos
    if pendientes==0:
        GOLD={(r.pregunta,r.mcid):int(r.relevancia_0_1_2) for r in qrels.itertuples(index=False)}
        def dcg(vals,k=5):
            return sum((2**v-1)/math.log2(i+2) for i,v in enumerate(vals[:k]))
        metricas=[]
        for q in consultas:
            gold={i:GOLD[(q,i)] for i in ids}
            n_relevantes=sum(x>=1 for x in gold.values())
            idcg=dcg(sorted(gold.values(),reverse=True))
            for m in ['A_tfidf','B_embeddings','C_hibrido_rrf','D_hibrido_rdf']:
                rels=[gold[i] for i in obtener_orden(q,m)]
                hits=sum(x>=1 for x in rels[:5])
                primera=next((i+1 for i,x in enumerate(rels[:10]) if x>=1),None)
                metricas.append({'pregunta':q,'metodo':m,'n_relevantes':n_relevantes,
                    'precision_at_5':hits/5,
                    'recall_at_5':hits/n_relevantes if n_relevantes else np.nan,
                    'ndcg_at_5':dcg(rels)/idcg if idcg else np.nan,
                    'mrr_at_10':1/primera if primera else 0.0})
        MET=pd.DataFrame(metricas)
        MET.to_csv(SALIDA/'metricas_por_consulta.csv',index=False,encoding='utf-8-sig')
        PROM=MET.groupby('metodo',as_index=False)[['precision_at_5','recall_at_5','ndcg_at_5','mrr_at_10']].mean()
        PROM.to_csv(SALIDA/'metricas_resumen.csv',index=False,encoding='utf-8-sig')
        metricas_creadas=True
        print('Métricas calculadas con 120 juicios humanos:')
        display(PROM)
    else:
        print(f'Evaluación pendiente: {validos}/120 juicios completos. No se calculan métricas.')
else:
    print('Evaluación pendiente: no hay juicios 09. No se calculan métricas.')

# Evitar métricas antiguas si se repite una ejecución con juicios incompletos.
if not metricas_creadas:
    for nombre in ('metricas_por_consulta.csv','metricas_resumen.csv'):
        (SALIDA/nombre).unlink(missing_ok=True)

Evaluación pendiente: 0/120 juicios completos. No se calculan métricas.


## 7. Consultas de prueba y salida ZIP automática

Puedes consultar cualquiera de los métodos con `buscar("tu pregunta", metodo="D_hibrido_rdf")`. Los resultados muestran ficha original, ranking y componentes; **no son respuestas generadas por IA**.

In [8]:
METODOS=['A_tfidf','B_embeddings','C_hibrido_rrf','D_hibrido_rdf']

def buscar(pregunta, metodo='D_hibrido_rdf', top_k=5):
    if metodo not in METODOS:
        raise ValueError(f'Método desconocido. Opciones: {METODOS}')
    if pregunta in consultas:
        qi=consultas.index(pregunta)
        lex=puntuaciones_lexicas(pregunta)
        sem=SCORES_SEMANTICOS[qi].astype(float)
    else:
        lex=puntuaciones_lexicas(pregunta)
        sem=np.asarray(modelo.encode([pregunta],convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False),dtype=np.float32)[0]@emb_docs.T
    fusion=puntuaciones_rrf(lex,sem)
    rdf,aporte=puntuaciones_rdf(fusion)
    scores=dict(zip(METODOS,[lex,sem,fusion,rdf]))[metodo]
    resultados=[]
    for puesto,i in enumerate(ordenar(scores)[:top_k],1):
        d=catalogo.iloc[i]
        resultados.append({'puesto':puesto,'mcid':ids[i],'titulo':d['titulo'],'ficha':d['ficha'],
                            'puntaje':round(float(scores[i]),6),
                            'tfidf':round(float(lex[i]),4),
                            'coseno_semantico':round(float(sem[i]),4),
                            'aporte_vecindad_rdf':round(float(aporte[i]),4)})
    return pd.DataFrame(resultados)

for pregunta in ['¿Qué obras catalogadas tratan sobre el aymara?',
                 '¿Qué textos coloniales están vinculados a la evangelización?']:
    print('\nConsulta:',pregunta)
    display(buscar(pregunta,top_k=5)[['puesto','titulo','mcid','puntaje','ficha']])

INFORME={
 'etapa':10,'generado_utc':datetime.now(timezone.utc).isoformat(),
 'fuentes':['etapa7.zip','etapa8.zip']+(['etapa9.zip/CSV' ] if qrels is not None else []),
 'n_fichas':len(ids),'n_consultas':len(consultas),'n_relaciones_especificas':len(relaciones),
 'n_filas_ranking':len(RANK),'n_emb_dimensiones':int(emb_docs.shape[1]),
 'modelo_semantico':MODELO_ID,
 'metodos':METODOS,'rrf_k':RRF_K,'bonificacion_rdf':BONUS_RDF,
 'semillas_grafo':TOP_SEMILLAS,'etiquetas_validas':validos,'etiquetas_pendientes':pendientes,
 'origen_juicios':fuente_juicios,
 'metricas_calculadas':metricas_creadas,
 'advertencias':[
    'Los embeddings se calculan sobre metadatos, no sobre los documentos íntegros',
    'La expansión RDF modela co-membresía catalográfica, no hechos históricos',
    'Las fechas de catálogo no se convierten automáticamente en fechas de publicación',
    'Cambios de ranking no implican mejora de relevancia',
    'Las métricas requieren los 120 juicios humanos completos'
 ]
}
(SALIDA/'informe_etapa10.json').write_text(json.dumps(INFORME,ensure_ascii=False,indent=2),encoding='utf-8')
DESTINO=DIR_OUTPUT/'etapa10.zip'
with zipfile.ZipFile(DESTINO,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SALIDA.iterdir()):
        if f.is_file(): z.write(f,arcname=f.name)
print('\nZIP creado automáticamente:',DESTINO)
print('Estado:', 'evaluación completa' if metricas_creadas else f'pendiente de {pendientes} juicios de relevancia')


Consulta: ¿Qué obras catalogadas tratan sobre el aymara?


,puesto,titulo,mcid,puntaje,ficha
0,1,"Primera parte de los commentarios reales, que ...",MC0014755,1.040354,https://www.memoriachilena.gob.cl/602/w3-artic...
1,2,Vocabulario de la lengua aymara,MC0014761,1.034344,https://www.memoriachilena.gob.cl/602/w3-artic...
2,3,Extirpación de la idolatria del Piru,MC0014760,1.009934,https://www.memoriachilena.gob.cl/602/w3-artic...
3,4,Historia natural y moral de las Indias : en qu...,MC0014757,1.009532,https://www.memoriachilena.gob.cl/602/w3-artic...
4,5,Doctrina christiana y catecismo para la instru...,MC0014753,0.993204,https://www.memoriachilena.gob.cl/602/w3-artic...



Consulta: ¿Qué textos coloniales están vinculados a la evangelización?


,puesto,titulo,mcid,puntaje,ficha
0,1,Doctrina christiana y catecismo para la instru...,MC0014753,1.015543,https://www.memoriachilena.gob.cl/602/w3-artic...
1,2,"Primera parte de los commentarios reales, que ...",MC0014755,1.010006,https://www.memoriachilena.gob.cl/602/w3-artic...
2,3,Extirpación de la idolatria del Piru,MC0014760,1.010006,https://www.memoriachilena.gob.cl/602/w3-artic...
3,4,Parte primera de la Chrónica del Perú : que tr...,MC0014754,1.007955,https://www.memoriachilena.gob.cl/602/w3-artic...
4,5,Historia natural y moral de las Indias : en qu...,MC0014757,0.983582,https://www.memoriachilena.gob.cl/602/w3-artic...



ZIP creado automáticamente: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa10.zip
Estado: pendiente de 120 juicios de relevancia


## 8. Qué hacemos después

1. Comparar los rankings A/B/C/D con preguntas que realmente distingan recuperación lexical, semántica y relacional.
2. Completar los 120 juicios del notebook 09; volver a ejecutar 10 para obtener métricas reales.
3. Analizar consultas donde los embeddings recuperan documentación útil que TF-IDF no prioriza, o donde el grafo introduce ruido.
4. Solo después elegir un método para añadir generación controlada por un LLM y citas a las fichas, sin afirmar contenido textual que no se haya verificado.

**No publiques archivos PDF originales ni modelos descargados en GitHub.** Los resultados del experimento son artefactos locales y deben revisarse antes de una publicación académica.